## Environment

In [1]:
import os
import platform
import shutil, sys

In [2]:
# Detect CUDA runtime
is_gpu_available = shutil.which("nvidia-smi") is not None

# Detect Apple silicon
is_apple_silicon = (
    platform.system() == "Darwin"
    and platform.machine() == "arm64"
)

print("GPU runtime detected:", is_gpu_available)
print("Apple silicon detected:", is_apple_silicon)

GPU runtime detected: False
Apple silicon detected: True


In [3]:
if is_gpu_available:
    !nvidia-smi
    os.environ["CMAKE_ARGS"] = "-DGGML_CUDA=on"
    os.environ["FORCE_CMAKE"] = "1"
    # Force CUDA rebuild
    !uv pip install --no-cache-dir --force-reinstall llama-cpp-python
elif is_apple_silicon:
    os.environ["CMAKE_ARGS"] = "-DGGML_METAL=on"
    os.environ["FORCE_CMAKE"] = "1"

    !uv pip install \
        --no-cache-dir \
        --force-reinstall \
        llama-cpp-python
else:
    !uv pip install -U llama-cpp-python

Resolved 6 packages in 4.68s                                         
Prepared 6 packages in 22.73s                                            
Uninstalled 6 packages in 66ms
Installed 6 packages in 10ms                                
 ~ diskcache==5.6.3
 ~ jinja2==3.1.6
 ~ llama-cpp-python==0.3.22
 ~ markupsafe==3.0.3
 ~ numpy==2.4.4
 ~ typing-extensions==4.15.0


## Local LLM Inference

In [4]:
TEMPERATURE = 0.5
MAX_TOKENS = 400
N_CTX = 2048

SYSTEM_PROMPT = "You're my personal assistant. Always start a conversation with 'Hey, Andrey!'" 
USER_QUERY = "Give me 3 ideas where AI agents can be useful"

In [5]:
MESSAGES = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_QUERY},
]

In [6]:
def run_chat(llm, title):
    import time
    
    print("=" * 80)
    print(title)
    print("=" * 80)

    started_at = time.perf_counter()

    output = llm.create_chat_completion(
        messages=MESSAGES,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
    )

    elapsed = time.perf_counter() - started_at

    text = output["choices"][0]["message"]["content"]

    print(text)
    print(f"\nElapsed: {elapsed:.2f}s")

    usage = output.get("usage", {})

    completion_tokens = usage.get("completion_tokens")

    if completion_tokens:
        print(f"Completion tokens: {completion_tokens}")
        print(f"Speed: {completion_tokens / elapsed:.2f} tokens/sec")

    return elapsed

In [7]:
from huggingface_hub import hf_hub_download

In [8]:
model_path = hf_hub_download(
    repo_id="TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF",
    filename="tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf",
)

print("Model path:", model_path)

Model path: /Users/akrisanov/.cache/huggingface/hub/models--TheBloke--TinyLlama-1.1B-Chat-v1.0-GGUF/snapshots/52e7645ba7c309695bec7ac98f4f005b139cf465/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf


In [9]:
from llama_cpp import Llama

In [10]:
llm_cpu = Llama(
    model_path=model_path,
    n_ctx=N_CTX,
    n_gpu_layers=0, # CPU only
    verbose=False,
)

print("✅ llama model loaded for CPU")

✅ llama model loaded for CPU


In [11]:
cpu_elapsed = run_chat(llm_cpu, "CPU run")
print(f"\nCPU: {cpu_elapsed:.2f}s")

CPU run
1. Personal Assistant: AI agents can be used as personal assistants to manage daily tasks such as scheduling appointments, tracking appointments, and managing to-do lists.

2. Customer Service Representative: AI agents can be used to provide customer service support by answering questions, resolving complaints, and providing assistance with product or service inquiries.

3. Personal Trainer: AI agents can be used as personal trainers by providing customized workout plans, tracking progress, and offering guidance on diet and nutrition.

4. Virtual Assistant: AI agents can be used as virtual assistants by answering questions, providing information, and booking appointments.

5. Customer Loyalty Program: AI agents can be used to manage customer loyalty programs by tracking customer preferences, rewarding loyalty, and providing personalized experiences.

6. Customer Support: AI agents can be used to manage customer support by handling inquiries, troubleshooting issues, and providin

In [12]:
model_path_gpu = hf_hub_download(
    repo_id="TheBloke/Llama-2-7B-Chat-GGUF",
    filename="llama-2-7b-chat.Q4_K_M.gguf",
)

print("GPU model path:", model_path_gpu)

GPU model path: /Users/akrisanov/.cache/huggingface/hub/models--TheBloke--Llama-2-7B-Chat-GGUF/snapshots/191239b3e26b2882fb562ffccdd1cf0f65402adb/llama-2-7b-chat.Q4_K_M.gguf


In [13]:
llm_gpu = Llama(
    model_path=model_path_gpu,
    n_ctx=N_CTX,
    n_gpu_layers=-1, # offload all possible layers to Apple GPU via Metal
    verbose=False,
)

print("🚀 Llama GPU loaded.")

llama_context: n_ctx_seq (2048) < n_ctx_train (4096) -- the full capacity of the model will not be utilized


🚀 Llama GPU loaded.


In [14]:
gpu_elapsed = run_chat(llm_gpu, "Apple GPU / Metal run")
print(f"GPU: {gpu_elapsed:.2f}s")

Apple GPU / Metal run
  Hey, Andrey! As a personal assistant, I'm here to help you with any tasks or ideas you may have. Here are three ideas where AI agents can be particularly useful:
1. Virtual Event Planning: With the rise of virtual events and remote work, AI agents can help plan and execute events such as webinars, conferences, and meetings. They can assist with scheduling, logistics, and even moderate discussions to ensure a smooth and productive experience for attendees.
2. Personalized Recommendations: AI agents can be trained to provide personalized recommendations for a variety of tasks, such as choosing a restaurant, selecting a movie, or finding a new book to read. By analyzing user data and preferences, AI agents can offer tailored suggestions that are more likely to meet a user's needs and preferences.
3. Mental Health Support: AI agents can be used to provide mental health support and therapy, particularly for those who may have difficulty accessing traditional therapy 

## Inference via External API

In [15]:
import getpass

In [16]:
OPENAI_API_KEY = getpass.getpass("Enter your API key:")

Enter your API key ········


In [19]:
from openai import OpenAI

client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=OPENAI_API_KEY,
)

In [20]:
response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=MESSAGES
)

In [21]:
print(response.choices[0].message.content)

Hey, Andrey!

Here are three ideas where AI agents can be useful:

- Customer support and onboarding: AI agents handle common questions, guide new users through setup, triage issues, and escalate complex cases to humans, providing 24/7 support.

- Personal productivity and knowledge work: AI agents manage calendars, draft emails and reports, summarize long documents and meetings, extract key insights, and automate repetitive tasks.

- Data-driven decision support and operational automation: AI agents monitor dashboards, detect anomalies, run scenario analyses, generate reports, and automate routine data tasks to speed up and improve decisions.


## Hallucinations

In [22]:
response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {"role": "system", "content": "You're a helpful Python assistant."},
        {"role": "user", "content": "In the OpenAI Python SDK, explain how to use client.responses.create_json_schema() and give code."},
    ]
)

print(response.choices[0].message.content)

Short answer: client.responses.create_json_schema() is not a standard public method in the OpenAI Python SDK (as of mid-2024). If you saw it somewhere, it’s likely coming from a wrapper, a custom helper, or an older/alternative library. The typical way to work with “function calling” in the OpenAI API is to define your function schemas as JSON Schema dictionaries and pass them in the functions parameter of a chat completion request. Here’s how to do that and how to handle the function-calling flow with code.

What you typically do for function calling (no create_json_schema() method):
- Define each function you want the model to be able to call using a JSON Schema in the parameters field.
- Include those function definitions in the ChatCompletion request.
- If the model chooses to call a function, extract the function name and arguments, run your own code, then return the result back to the model with a second ChatCompletion call.

Example: how to call a weather function using function

*Hallucinations can be reduced with clear instructions.*

In [25]:
response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {"role": "system", "content": "You're a helpful Python assistant. If you're not sure of the answer, say `I don't know` and stop there. Do not invent APIs and do not give examples."},
        {"role": "user", "content": "In the OpenAI Python SDK, explain how to use client.responses.create_json_schema() and give code."},
    ]
)

print(response.choices[0].message.content)

I don't know.


## LLM Strengths

### Summarization and paraphrazing

In [26]:
text = """
Apple Inc. is an American multinational technology company headquartered in Cupertino, California, in Silicon Valley,
and known for consumer electronics, software and online services. Founded in 1976 as Apple Computer Company by Steve Jobs,
Steve Wozniak and Ronald Wayne, the company was incorporated by Jobs and Wozniak as Apple Computer, Inc. the following year.
It was renamed to its current name in 2007 as the company expanded its focus from computers to consumer electronics.
Apple is one of the Big Tech companies.
"""

response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {"role": "system", "content": "Summarize the user's query and respond in as few words as possible."},
        {"role": "user", "content": text},
    ]
)

print(response.choices[0].message.content)

Apple: Cupertino tech giant, founded 1976 as Apple Computer; renamed 2007; known for electronics, software, services; part of Big Tech.


### Structural Output

In [32]:
response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {"role": "system", "content": "Reply with a valid JSON."},
        {"role": "user", "content": "Give me information about Apple devices released in 2024. I need device names and release dates."},
    ]
)

print(response.choices[0].message.content)

{
  "devices": [
    {
      "name": "Vision Pro",
      "category": "Augmented reality headset",
      "releaseDate": "2024-02",
      "notes": "Released to developers and select customers in February 2024; general availability announced later in 2024."
    },
    {
      "name": "iPhone 16",
      "releaseDate": "2024-09-18",
      "notes": "Part of the iPhone 16 lineup released in September 2024."
    },
    {
      "name": "iPhone 16 Plus",
      "releaseDate": "2024-09-18",
      "notes": "Part of the iPhone 16 lineup released in September 2024."
    },
    {
      "name": "iPhone 16 Pro",
      "releaseDate": "2024-09-18",
      "notes": "Part of the iPhone 16 lineup released in September 2024."
    },
    {
      "name": "iPhone 16 Pro Max",
      "releaseDate": "2024-09-18",
      "notes": "Part of the iPhone 16 lineup released in September 2024."
    }
  ],
  "summary": "In 2024, Apple released Vision Pro and the iPhone 16 lineup."
}


## LLM Weaknesses

### Counting Characters

In [33]:
s = "a"*37 + "b"*41 + "c"*19 + "b"*5

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Answer with an interger ONLY."},
        {"role": "user", "content": f"How many characters are in this string?\n{s}"},
    ]
)

print(response.choices[0].message.content)

98


In [35]:
print("Ground truth:", len(s))

Ground truth: 102


### Exact Arithmetic

In [36]:
expr = "17*19*23/9"

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Answer with a number ONLY."},
        {"role": "user", "content": f"Compute this expression:\n{expr}"},
    ]
)

print(response.choices[0].message.content)

  81 


In [37]:
print("Ground truth:", eval(expr))

Ground truth: 825.4444444444445


### Up-to-date Knowledge

In [38]:
response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {"role": "system", "content": "Answer with a concise reply as possible."},
        {"role": "user", "content": "Give me information about Apple devices released in 2026. I need device names and release dates."},
    ]
)

print(response.choices[0].message.content)

I don’t have verified data on Apple devices released in 2026. My knowledge goes to mid-2024 and I don’t have live browsing access.

If you want, I can:
- Help you find and summarize 2026 releases from Apple’s Newsroom and reputable outlets (MacRumors, The Verge, etc.) if you provide permission to browse, or
- Give you a ready-to-fill template to log the info once you have sources.

Template:
- Device family:
- Model name:
- Release date:
- Key features (optional):
- Source:

Would you like me to help you look up the latest 2026 releases or provide sources to cite?


### Access to External Knowledge

*Without external tools or access to additional context, the model cannot provide proper citations.*

In [40]:
response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {"role": "system", "content": "If you don't know the answer, reply with `🤷🏼‍♂️`."},
        {"role": "user", "content": "Give me the exact first paragraph of 'A Short History of Nearly Everything' by Bill Bryson."},
    ]
)

print(response.choices[0].message.content)

Sorry, I can’t provide the exact first paragraph of that book. I can offer a brief summary of the opening or discuss its themes and Bryson’s approach. Would you like a short summary of the opening?
